In [ ]:
import json
from pathlib import Path
from collections import Counter

In [ ]:
from pathlib import Path

print(Path.cwd())

In [ ]:
print("HDB files:", list(Path("../data/hdb").glob("*.geojson")))

In [ ]:
print("URA files:", list(Path("../data/ura").glob("*.geojson")))

In [ ]:
# Copy the full file paths from VS Code Explorer
hdb_path = Path("../data/hdb/HDBUnderConstructionBuildings.geojson")
ura_path = Path("../data/ura/MasterPlan2025LandUseLayer.geojson")

def inspect_geojson(file_path, dataset_name):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    features = data.get("features", [])

    print(f"\n{'=' * 55}")
    print(dataset_name)
    print("=" * 55)

    print("Number of features:", len(features))
    print("GeoJSON type:", data.get("type"))

    if features:
        # Collect property names across ALL features
        all_fields = sorted({
            key
            for feature in features
            for key in feature.get("properties", {}).keys()
        })

        geometry_types = Counter(
            feature.get("geometry", {}).get("type", "Missing")
            if feature.get("geometry") else "Missing"
            for feature in features
        )

        print("\nAll available fields:")
        print(all_fields)

        print("\nGeometry types:")
        print(dict(geometry_types))

        print("\nFirst record's properties:")
        for key, value in features[0].get("properties", {}).items():
            print(f"  {key}: {str(value)[:120]}")

    return data


hdb_data = inspect_geojson(
    hdb_path,
    "HDB Under Construction Buildings"
)

ura_landuse_data = inspect_geojson(
    ura_path,
    "URA Master Plan 2025 Land Use"
)

In [ ]:
import pandas as pd

# URA land-use category counts
ura_categories = pd.Series(
    [
        feature["properties"].get("LU_DESC")
        for feature in ura_landuse_data["features"]
    ]
)

print("URA land-use categories:")
print(ura_categories.value_counts(dropna=False).to_string())

# HDB record-update years
hdb_update_years = pd.Series(
    [
        str(feature["properties"].get("FMEL_UPD_D", ""))[:4]
        for feature in hdb_data["features"]
    ]
)

print("\nHDB record-update years:")
print(
    hdb_update_years.value_counts(dropna=False)
    .sort_index()
    .to_string()
)

In [ ]:
import json
from pathlib import Path

# Search for the planning-area GeoJSON
filename = "MasterPlan2025PlanningAreaBoundaryNoSea.geojson"

# Search the current folder and its parent
matches = list(Path.cwd().rglob(filename))

if not matches:
    matches = list(Path.cwd().parent.rglob(filename))

# Ensure exactly one file was found
assert len(matches) == 1, (
    f"Expected one GeoJSON file, found {len(matches)}: {matches}"
)

# Load the GeoJSON
planning_area_path = matches[0]

with open(planning_area_path, "r", encoding="utf-8") as f:
    ura_data = json.load(f)

print("File found:", planning_area_path)
print("URA planning areas loaded:", len(ura_data["features"]))

In [ ]:

import pandas as pd
from shapely.geometry import shape
from shapely.strtree import STRtree

# Reuse the URA planning-area GeoJSON loaded earlier.
# If ura_data is not available in this notebook,
# load it from your existing planning-area GeoJSON file.

planning_features = ura_data["features"]

planning_geometries = [
    shape(feature["geometry"])
    for feature in planning_features
]

planning_names = [
    feature["properties"]["PLN_AREA_N"]
    for feature in planning_features
]

# Spatial index for efficient geographic matching
tree = STRtree(planning_geometries)

records = []

for feature in hdb_data["features"]:

    record = feature["properties"].copy()
    geometry = shape(feature["geometry"])

    # A point guaranteed to lie within the building footprint
    point = geometry.representative_point()

    # Find the URA planning area containing the point
    matches = tree.query(point, predicate="intersects")

    record["planning_area"] = (
        planning_names[int(matches[0])]
        if len(matches) > 0
        else None
    )

    # Preserve geometry as WKT
    record["geometry_wkt"] = geometry.wkt

    records.append(record)

hdb_development_df = pd.DataFrame(records)

print("Original buildings:", len(hdb_data["features"]))
print("Processed buildings:", len(hdb_development_df))
print("Unmatched buildings:",
      hdb_development_df["planning_area"].isna().sum())

assert len(hdb_development_df) == len(hdb_data["features"])

display(
    hdb_development_df.groupby("planning_area")
    .size()
    .reset_index(name="recorded_building_footprints")
    .sort_values("recorded_building_footprints", ascending=False)
)


In [ ]:
hdb_development_summary = (
    hdb_development_df
    .groupby("planning_area")
    .agg(
        building_footprints=("OBJECTID", "count"),
        latest_record_update=("FMEL_UPD_D", "max")
    )
    .reset_index()
    .sort_values("building_footprints", ascending=False)
)

display(hdb_development_summary)

print(
    "Total building footprints:",
    hdb_development_summary["building_footprints"].sum()
)